In [2]:
import cv2
import urllib.request
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
model_path = 'gesture_recognizer.task'
if not os.path.exists(model_path):
    print("Downloading model...")
    url = 'https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'
    urllib.request.urlretrieve(url, model_path)
    print("Download complete.")

In [4]:
action_map = {
    "Closed_Fist": "Shield",
    "Open_Palm": "Damage",
    "Thumb_Up": "Buff",
    "Victory": "Debuff",
    "Pointing_Up": "Magic"
}

# 3. Configure the Recognizer
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(base_options=base_options)
recognizer = vision.GestureRecognizer.create_from_options(options)

I0000 00:00:1778241246.815787   40926 gl_context.cc:369] GL version: 2.1 (2.1 ATI-7.1.6), renderer: AMD Radeon Pro 5500M OpenGL Engine
W0000 00:00:1778241246.816783   40926 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
I0000 00:00:1778241246.819015   40926 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778241246.853387   41492 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778241246.879719   41497 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778241246.881313   41496 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling 

In [ ]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    # Convert OpenCV frame (BGR) to MediaPipe Image (RGB)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Run the model
    recognition_result = recognizer.recognize(mp_image)
    
    current_action = "None"
    
    # Parse the output
    if recognition_result.gestures:
        top_gesture = recognition_result.gestures[0][0].category_name
        current_action = action_map.get(top_gesture, "None")
        
    # Display the result
    cv2.putText(frame, f"Game Action: {current_action}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Local Gesture Recognition', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1778241248.990875   41498 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


: 